In [1]:
# ENVIRONMENT CUTADAPT
# KERNEL R

In [2]:
#### CUTADAPT
##### Primers removal 
##### ITS amplicon sequencing by Illumin MiSeq 2x300

In [13]:
# 1. Load packages-----------


# Check Cutadapt availability


Sys.which("cutadapt")
library(ShortRead)
library(Biostrings)
library(dplyr)

cutadapt 
"/home/tatiana/Programas/anaconda3/envs/cutadapt/bin/cutadapt"


Attaching package: ‘dplyr’


The following object is masked from ‘package:ShortRead’:

    id


The following objects are masked from ‘package:GenomicAlignments’:

    first, last


The following object is masked from ‘package:Biobase’:

    combine


The following object is masked from ‘package:matrixStats’:

    count


The following objects are masked from ‘package:GenomicRanges’:

    intersect, setdiff, union


The following objects are masked from ‘package:Biostrings’:

    collapse, intersect, setdiff, setequal, union


The following object is masked from ‘package:GenomeInfoDb’:

    intersect


The following object is masked from ‘package:XVector’:

    slice


The following objects are masked from ‘package:IRanges’:

    collapse, desc, intersect, setdiff, slice, union


The following objects are masked from ‘package:S4Vectors’:

    first, intersect, rename, setdiff, setequal, union


The following objects are masked from ‘package:BiocGenerics’:

    combine, intersect, setd

In [14]:
# 1.1. Check package versions-----------

R.version.string
packageVersion("ShortRead")
packageVersion("Biostrings")
packageVersion("dplyr")

[1] "R version 4.1.1 (2021-08-10)"

[1] ‘1.50.0’

[1] ‘2.62.0’

[1] ‘1.0.7’

In [6]:
# 2. Define input and output paths-----------

# Directory containing the original paired-end FASTQ files.
# Change this path if your sequencing files are stored elsewhere.

# set up working directory

input_dir <-"/home/tatiana/PROYECTOS/repository/sequences" 
input_dir

# Output directory for trimmed reads, reports and logs.

output_dir <- file.path(input_dir, "its_results")

# Primer sequences, written 5' to 3'. # Replace by the pairs of primers

ITSF <- "CTTGGTCATTTAGAGGAAGTAA"
ITS2 <- "GCTGCGTTCTTCATCGATGC"


# Number of reads used to inspect primer orientations.

primer_inspection_reads <- 10000L

# TRUE: discard a read pair if either read lacks its expected primer.
# FALSE: retain untrimmed pairs as well.


discard_untrimmed <- TRUE

[1] "/home/tatiana/PROYECTOS/repository/sequences"

In [7]:
# 3. Create output directories-----------

cutadapt_dir <- file.path(output_dir, "cutadapt")
reports_dir <- file.path(output_dir, "reports")
logs_dir <- file.path(output_dir, "logs")

dir.create(
  cutadapt_dir,
  recursive = TRUE,
  showWarnings = FALSE
)

dir.create(
  reports_dir,
  recursive = TRUE,
  showWarnings = FALSE
)

dir.create(
  logs_dir,
  recursive = TRUE,
  showWarnings = FALSE
)


if (!dir.exists(input_dir)) {
  stop("Input directory not found. Check input_dir.")
}

In [8]:
# 4. Check dependencies and Cutadapt-----------

# Check that the required R packages are available.

required_packages <- c(
  "ShortRead",
  "Biostrings"
)


missing_packages <- required_packages[
  !vapply(
    required_packages,
    requireNamespace,
    logical(1),
    quietly = TRUE
  )
]


if (length(missing_packages) > 0) {
  stop(
    "Missing R packages: ",
    paste(missing_packages, collapse = ", "),
    ". Install them before running this script."
  )
}


# Locate the Cutadapt executable.


cutadapt_path <- Sys.which("cutadapt")


if (!nzchar(cutadapt_path)) {
  stop(
    "Cutadapt was not found in PATH. ",
    "Check that the executable is installed and accessible."
  )
}


# Retrieve the installed Cutadapt version.

cutadapt_version <- tryCatch(
  paste(
    system2(
      cutadapt_path,
      args = "--version",
      stdout = TRUE,
      stderr = TRUE
    ),
    collapse = " "
  ),
  error = function(e) {
    paste("Version check failed:", conditionMessage(e))
  }
)


# Record software versions and R session information.

version_lines <- c(
  paste(
    "Date:",
    format(Sys.time(), "%Y-%m-%d %H:%M:%S %Z")
  ),
  paste("R:", R.version.string),
  paste(
    "ShortRead:",
    as.character(packageVersion("ShortRead"))
  ),
  paste(
    "Biostrings:",
    as.character(packageVersion("Biostrings"))
  ),
  paste("Cutadapt executable:", cutadapt_path),
  paste("Cutadapt version:", cutadapt_version)
)


writeLines(
  version_lines,
  file.path(reports_dir, "software_versions.txt")
)


writeLines(
  capture.output(sessionInfo()),
  file.path(reports_dir, "sessionInfo.txt")
)

In [9]:
# 5. Identify and pair FASTQ files-----------

# Detect paired-end FASTQ files automatically.
# Expected suffixes:
# - _R1_001.fastq.gz
# - _R2_001.fastq.gz

fastq_R1 <- sort(
  list.files(
    input_dir,
    pattern = "_R1_001\\.fastq\\.gz$",
    full.names = TRUE
  )
)

fastq_R2 <- sort(
  list.files(
    input_dir,
    pattern = "_R2_001\\.fastq\\.gz$",
    full.names = TRUE
  )
)

if (length(fastq_R1) == 0 || length(fastq_R2) == 0) {
  stop(
    "No paired FASTQ files found. ",
    "Check the expected filename suffixes."
  )
}

# Extract sample keys from filenames.

sample_key_R1 <- sub(
  "_R1_001\\.fastq\\.gz$",
  "",
  basename(fastq_R1)
)


sample_key_R2 <- sub(
  "_R2_001\\.fastq\\.gz$",
  "",
  basename(fastq_R2)
)


if (
  anyDuplicated(sample_key_R1) ||
  anyDuplicated(sample_key_R2)
) {
  stop("Duplicated sample identifiers were found in FASTQ filenames.")
}

# Identify missing mate files.

missing_R2 <- setdiff(sample_key_R1, sample_key_R2)
missing_R1 <- setdiff(sample_key_R2, sample_key_R1)


if (length(missing_R2) > 0 || length(missing_R1) > 0) {
  stop(
    "FASTQ files are not correctly paired. ",
    "Check for missing R1 or R2 files."
  )
}


# Match R2 files to the order of R1 files.


fastq_R2 <- fastq_R2[
  match(sample_key_R1, sample_key_R2)
]

sample_ids <- sample_key_R1


# Create an internal sample manifest.
# The manifest is used to associate input and output files.

sample_manifest <- data.frame(
  Sample = sample_ids,
  Input_R1 = fastq_R1,
  Input_R2 = fastq_R2,
  stringsAsFactors = FALSE
)

In [10]:
# 6. Inspect primer orientations-----------

# Generate the four possible orientations of each primer.

primer_orientations <- function(primer) {


  primer_dna <- Biostrings::DNAString(primer)


  c(
    Forward = as.character(primer_dna),


    Complement = as.character(
      Biostrings::complement(primer_dna)
    ),


    Reverse = as.character(
      Biostrings::reverse(primer_dna)
    ),


    Reverse_complement = as.character(
      Biostrings::reverseComplement(primer_dna)
    )
  )
}

# Count reads containing each primer orientation.

count_orientations <- function(primer, reads) {

  orientations <- primer_orientations(primer)

  vapply(
    orientations,
    function(pattern) {

      sum(
        Biostrings::vcountPattern(
          pattern,
          reads,
          fixed = FALSE
        ) > 0
      )


    },
    numeric(1)
  )
}

# Read a limited number of reads from the first FASTQ pair.
# Streaming avoids loading an entire sequencing file into memory.

stream_R1 <- ShortRead::FastqStreamer(
  fastq_R1[1],
  n = primer_inspection_reads
)

reads_R1 <- ShortRead::yield(stream_R1)
close(stream_R1)

stream_R2 <- ShortRead::FastqStreamer(
  fastq_R2[1],
  n = primer_inspection_reads
)

reads_R2 <- ShortRead::yield(stream_R2)
close(stream_R2)

if (length(reads_R1) == 0 || length(reads_R2) == 0) {
  stop("The first FASTQ pair contains no reads.")
}


# Compare the number of records inspected.

reads_examined <- min(
  length(reads_R1),
  length(reads_R2)
)

if (length(reads_R1) != length(reads_R2)) {
  stop(
    "The first R1 and R2 files yielded different numbers of reads."
  )
}


# Count each primer orientation in each read direction.

primer_matrix <- rbind(
  ITSF_in_R1 = count_orientations(ITSF, ShortRead::sread(reads_R1)),
  ITSF_in_R2 = count_orientations(ITSF, ShortRead::sread(reads_R2)),
  ITS2_in_R1 = count_orientations(ITS2, ShortRead::sread(reads_R1)),
  ITS2_in_R2 = count_orientations(ITS2, ShortRead::sread(reads_R2))
)

primer_counts <- data.frame(
  Read_orientation = rownames(primer_matrix),
  as.data.frame(primer_matrix, check.names = FALSE),
  check.names = FALSE,
  row.names = NULL
)

primer_counts$Reads_examined <- reads_examined

primer_counts <- primer_counts[
  ,
  c(
    "Read_orientation",
    "Reads_examined",
    "Forward",
    "Complement",
    "Reverse",
    "Reverse_complement"
  )
]

write.csv(
  primer_counts,
  file.path(reports_dir, "primer_counts.csv"),
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

# Release memory used by the primer inspection.
rm(reads_R1, reads_R2, primer_matrix)
invisible(gc())


In [11]:
# 7. Trim primers with Cutadapt-----------

# Prepare output FASTQ paths.

cut_R1 <- file.path(
  cutadapt_dir,
  basename(fastq_R1)
)

cut_R2 <- file.path(
  cutadapt_dir,
  basename(fastq_R2)
)

# Prepare one log file per sample.

log_files <- file.path(
  logs_dir,
  paste0(sample_ids, "_cutadapt.log")
)

# Initialise the processing summary.

cutadapt_summary <- data.frame(
  Sample = sample_ids,
  Status = NA_integer_,
  Output_R1 = cut_R1,
  Output_R2 = cut_R2,
  Log_file = log_files,
  stringsAsFactors = FALSE
)

# Run Cutadapt for each paired-end sample.

for (i in seq_along(sample_ids)) {
  args <- c(
    "-g", ITSF,
    "-G", ITS2
  )

  if (discard_untrimmed) {
    args <- c(
      args,
      "--discard-untrimmed",
      "--pair-filter=any"
    )
  }

  args <- c(
    args,
    "-o", shQuote(cut_R1[i]),
    "-p", shQuote(cut_R2[i]),
    shQuote(fastq_R1[i]),
    shQuote(fastq_R2[i])
  )

  log_output <- tryCatch(
    suppressWarnings(
      system2(
        cutadapt_path,
        args = args,
        stdout = TRUE,
        stderr = TRUE
      )
    ),
    error = function(e) {
      structure(
        paste(
          "Cutadapt execution error:",
          conditionMessage(e)
        ),
        status = 1L
      )
    }
  )
  exit_status <- attr(log_output, "status")

  if (is.null(exit_status)) {
    exit_status <- 0L
  }

  writeLines(
    as.character(log_output),
    con = log_files[i]
  )

  cutadapt_summary$Status[i] <- as.integer(exit_status)
}

# Verify output files and summarise processing status.

cutadapt_summary$Output_R1_exists <- file.exists(cut_R1)
cutadapt_summary$Output_R2_exists <- file.exists(cut_R2)

cutadapt_summary$Success <- (
  cutadapt_summary$Status == 0L &
    cutadapt_summary$Output_R1_exists &
    cutadapt_summary$Output_R2_exists
)

In [16]:
# 8. Export Cutadapt summary-----------

write.csv(
  cutadapt_summary,
  file = file.path(reports_dir, "cutadapt_summary.csv"),
  row.names = FALSE,
  fileEncoding = "UTF-8"
)

# Export a list of samples that did not complete successfully.

failed_samples <- cutadapt_summary %>%
  dplyr::filter(!.data$Success)

write.csv(
  failed_samples,
  file = file.path(reports_dir, "cutadapt_failed_samples.csv"),
  row.names = FALSE,
  fileEncoding = "UTF-8"
)